 # Predicting Car Prices 

## 1. Business Problem

We work for a used-car marketplace. A seller uploads a car (Make, Colour, Doors, Odometer) and we want to suggest a fair **Price**.

- **Why regression?** The target `Price` is a *continuous number*, not a category.
- **Business use:** auto-suggest listing prices, flag over/under-priced cars, estimate inventory value.

## 2. Explore the Data

Before any model: look at the data. Shape, features, target, and column types.

In [30]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
print(f"Using Scikit-Learn version: {sklearn.__version__}")

Using Scikit-Learn version: 1.9.0


In [31]:
# Load the clean version first
car_sales = pd.read_csv("../data/car-sales-extended.csv")
car_sales.head()

,Make,Colour,Odometer (KM),Doors,Price
0,Honda,White,35431,4,15323
1,BMW,Blue,192714,5,19943
2,Honda,White,84714,4,28343
3,Toyota,White,154365,4,13434
4,Nissan,Blue,181577,3,14043


In [32]:
car_sales.shape      # rows (samples), columns (features)

(1000, 5)

In [33]:
car_sales.dtypes     # which columns are NOT numbers?

Make               str
Colour             str
Odometer (KM)    int64
Doors            int64
Price            int64
dtype: object

**Observations:** `Make` and `Colour` are `object` (text). `Doors`, `Odometer (KM)`, `Price` are numeric.
Target = `Price`. Features = everything else. The text columns are the problem — a model can't multiply 'Toyota'.

**Ask students:** Which columns aren't numbers? Is `Doors` a quantity or a category?

## 3. Prepare the Data

### A. Separate X and y

X = inputs the model sees. y = the answer it predicts.

In [34]:
# X = features (everything except the answer), y = target
X = car_sales.drop("Price", axis=1)
y = car_sales["Price"]
X.head()

,Make,Colour,Odometer (KM),Doors
0,Honda,White,35431,4
1,BMW,Blue,192714,5
2,Honda,White,84714,4
3,Toyota,White,154365,4
4,Nissan,Blue,181577,3


### B. Show *why* raw data fails (this cell errors on purpose)

Let it fail — it's the best teaching moment in the project.

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)   # ValueError: could not convert string to float: 'Toyota'

ValueError: could not convert string to float: 'Toyota'

`could not convert string to float` — the model is math, math needs numbers. **This is why the whole 'Prepare' phase exists.**

### C. Handle Missing Values

Switch to the realistic version (same data, with holes = `NaN`). Models refuse to train with missing values, so we **drop** or **fill** (impute).

**Rule:** the target is sacred — if a row is missing its `Price`, drop it (never invent the answer). Missing *features* we fill intelligently.

In [36]:
car_sales_missing = pd.read_csv("../data/car-sales-extended-missing-data.csv")

# How many holes, and where?
car_sales_missing.isna().sum()

Make             49
Colour           50
Odometer (KM)    50
Doors            50
Price            50
dtype: int64

In [37]:
# Rule: never invent the answer. Drop rows missing the target (Price).
car_sales_missing.dropna(subset=["Price"], inplace=True)

# Re-split into X and y on the cleaned data
X = car_sales_missing.drop("Price", axis=1)
y = car_sales_missing["Price"]

Different column types deserve different fills:
- **Text** (`Make`, `Colour`): constant word `"missing"` — treat unknown as its own category.
- **Doors**: constant `4` — most cars have 4 doors.
- **Odometer** (a measurement): the column **mean**.

In [38]:
from sklearn.impute import SimpleImputer

cat_imputer  = SimpleImputer(strategy="constant", fill_value="missing")  # Make, Colour
door_imputer = SimpleImputer(strategy="constant", fill_value=4)          # Doors
num_imputer  = SimpleImputer(strategy="mean")                            # Odometer (KM)

### D. Convert Categories to Numbers (One-Hot Encoding)

Don't map Toyota=1, Honda=2, BMW=3 — that invents a fake ordering the model will misuse.
**One-Hot Encoding** makes each category its own 0/1 column. Quick illustration with pandas:

Sample   Honda toyata BMW 
0         1    0      0
1         0    1      0
2         0    0       1

In [40]:
# Illustration of what one-hot encoding produces
dummies = pd.get_dummies(data=car_sales[["Make", "Colour", "Doors"]], dtype=float)
dummies.head()

,Doors,Make_BMW,Make_Honda,Make_Nissan,Make_Toyota,Colour_Black,Colour_Blue,Colour_Green,Colour_Red,Colour_White
0,4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,5,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,3,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [41]:
car_sales["Doors"]= car_sales["Doors"].astype(object)
dummies = pd.get_dummies(data=car_sales[["Make", "Colour", "Doors"]], dtype=float)
dummies.head()

,Make_BMW,Make_Honda,Make_Nissan,Make_Toyota,Colour_Black,Colour_Blue,Colour_Green,Colour_Red,Colour_White,Doors_3,Doors_4,Doors_5
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


One text column exploded into several 0/1 columns. We'll do this properly with sklearn's `OneHotEncoder` inside the pipeline (so it's reusable on new data).

### E. Apply Transformations with ColumnTransformer

Different columns need different recipes. **ColumnTransformer** routes each transformer to the right columns in one object.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ["Make", "Colour", "Doors"]

one_hot = OneHotEncoder()

transformer = ColumnTransformer([("one_hot", 
                                  one_hot, 
                                  categorical_features)],
                                remainder="passthrough",
                                sparse_threshold=0) # return a sparse matrix or not

transformed_X_missing = transformer.fit_transform(X_missing)
transformed_X_missing

In [45]:
from sklearn.compose import ColumnTransformer

categorical_features = ["Make", "Colour"]
door_feature         = ["Doors"]
numerical_feature    = ["Odometer (KM)"]

# Route each group of columns to the right imputer
imputer = ColumnTransformer([
    ("cat_imputer",  cat_imputer,  categorical_features),
    ("door_imputer", door_imputer, door_feature),
    ("num_imputer",  num_imputer,  numerical_feature)])

In [47]:
car_sales_missing.isna().sum()

Make             47
Colour           46
Odometer (KM)    48
Doors            47
Price             0
dtype: int64

## 4. Train/Test Split & Data Leakage

Split into a **training set** (model learns) and a **test set** (model is graded, never seen). Typically 80/20.

**Data leakage:** if the imputer learns its mean from the *whole* dataset (including test rows), test info leaks into preprocessing and the score becomes a lie.

**The rule:** preprocessing must `fit` on the training set only.
- `fit_transform(X_train)` → **learn** the fills **and** apply them — train only.
- `transform(X_test)` → **apply** what was learned — test only.

In [52]:
from sklearn.model_selection import train_test_split

np.random.seed(42)  # reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# fit_transform on TRAIN: the imputer LEARNS the means/fills here...
filled_X_train = imputer.fit_transform(X_train)

# transform ONLY on TEST: it just APPLIES what it learned from train.
filled_X_test  = imputer.transform(X_test)
filled_X_train[:3]

array([['Honda', 'White', 4.0, 71934.0],
       ['Toyota', 'Red', 4.0, 162665.0],
       ['Honda', 'White', 4.0, 42844.0]], dtype=object)

## 5. Build a Preprocessing Pipeline

Doing impute -> encode -> split by hand is fragile. A **Pipeline** chains steps into one object, and `.fit()` automatically fits every step on training data only.

**This is how preprocessing is done in real projects.**

In [53]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Load fresh and drop rows with no target
data = pd.read_csv("../data/car-sales-extended-missing-data.csv")
data.dropna(subset=["Price"], inplace=True)

# Each column-group gets a mini-pipeline: fill, then (if text) one-hot
categorical_features = ["Make", "Colour"]
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))])

door_feature = ["Doors"]
door_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=4))])

numeric_features = ["Odometer (KM)"]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean"))])

# Combine them: route each group to its mini-pipeline
preprocessor = ColumnTransformer(transformers=[
    ("cat",  categorical_transformer, categorical_features),
    ("door", door_transformer,        door_feature),
    ("num",  numeric_transformer,     numeric_features)])

# Bolt the model on the end -> ONE object that does everything
model = Pipeline(steps=[("preprocessor", preprocessor),
                        ("model", RandomForestRegressor(n_jobs=-1))])
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('door', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3

## 6. Train the Model

`.fit()` is where learning happens — it walks every training example and adjusts the model so its price guesses get closer to the truth. Because `model` is the whole pipeline, this one call imputes, encodes, **and** trains — on training data only.

In [54]:
X = data.drop("Price", axis=1)
y = data["Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(X_train, y_train)   # preprocess + learn, in one call

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['Make','Colour','Odometer (KM)','Doors']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('door', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the o

## 7. Evaluate the Model

For regression: **MAE, MSE, RMSE, R²**. `.score()` on a regressor returns **R²**.

In [57]:
# Quick health check - for a regressor, .score() returns R^2
model.score(X_test, y_test)

0.22188417408787864

In [58]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_preds)
mse  = mean_squared_error(y_test, y_preds)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_preds)

print(f"R2:   {r2:.3f}")
print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

R2:   0.222
MAE:  5713.17
MSE:  52102396.83
RMSE: 7218.20


- **MAE** — average dollar error. Readable; doesn't punish big misses extra.
- **MSE** — average squared error (dollars²). Punishes big errors; not human-readable.
- **RMSE** — root of MSE, back in dollars. Readable *and* punishes big errors. A big RMSE–MAE gap = a few large errors.
- **R²** — fraction of price variation explained. 1=perfect, 0=no better than guessing the mean price. Pair it with MAE/RMSE.

## 8. Improve the Model

Never marry your first model. Two levers: try a different **algorithm**, and tune **hyperparameters**.

In [59]:
from sklearn.linear_model import LinearRegression

# Same preprocessor, different model on the end
lin_model = Pipeline(steps=[("preprocessor", preprocessor),
                            ("model", LinearRegression())])
lin_model.fit(X_train, y_train)
print(f"LinearRegression R2: {lin_model.score(X_test, y_test):.3f}")
print(f"RandomForest     R2: {model.score(X_test, y_test):.3f}")

LinearRegression R2: 0.253
RandomForest     R2: 0.222


In [60]:
from sklearn.model_selection import GridSearchCV

# Note the double-underscore path: pipelineStep__subStep__param
pipe_grid = {
    "preprocessor__num__imputer__strategy": ["mean", "median"],
    "model__n_estimators": [100, 1000],
    "model__max_depth": [None, 5],
    "model__max_features": ["sqrt"],
    "model__min_samples_split": [2, 4],
}

gs_model = GridSearchCV(model, pipe_grid, cv=5, verbose=2)
gs_model.fit(X_train, y_train)
gs_model.score(X_test, y_test)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_split=2, model__n_estimators=100, preprocessor__num__imputer__strategy=mean; total time=   1.0s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_split=2, model__n_estimators=100, preprocessor__num__imputer__strategy=mean; total time=   1.2s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_split=2, model__n_estimators=100, preprocessor__num__imputer__strategy=mean; total time=   1.5s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_split=2, model__n_estimators=100, preprocessor__num__imputer__strategy=mean; total time=   0.9s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_split=2, model__n_estimators=100, preprocessor__num__imputer__strategy=mean; total time=   1.5s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_sampl

0.2848784564026806

In [67]:
gs_model.best_params_

{'model__max_depth': 5,
 'model__max_features': 'sqrt',
 'model__min_samples_split': 2,
 'model__n_estimators': 100,
 'preprocessor__num__imputer__strategy': 'mean'}

## 9. Save the Model

Serialize the trained pipeline so the app can load and predict instantly. **pickle** (built-in) or **joblib** (faster for big models).

In [61]:
import pickle

# Save the WHOLE pipeline (preprocessing + model together)
pickle.dump(gs_model, open("car_price_model.pkl", "wb"))

loaded = pickle.load(open("car_price_model.pkl", "rb"))
print(f"Loaded model R2: {loaded.score(X_test, y_test):.3f}")

Loaded model R2: 0.285


In [62]:
from joblib import dump, load

dump(gs_model, "car_price_model.joblib")
loaded_joblib = load("car_price_model.joblib")
print(f"Loaded joblib model R2: {loaded_joblib.score(X_test, y_test):.3f}")

Loaded joblib model R2: 0.285


In [75]:
# A brand-new listing comes in - RAW data, exactly like the CSV.
# Note: Doors is given as 4.0 (float) so its dtype matches the training
# data, where Doors became float because that column contained missing values.
new_car = pd.DataFrame({
    "Make": ["Toyota"],
    "Colour": ["White"],
    "Odometer (KM)": [55000],
    "Doors": [4.0],
})
print(f"Suggested price: ${loaded.predict(new_car)[0]:,.0f}")

Suggested price: $19,054


Because we saved the *whole pipeline*, the loaded model imputes and encodes this raw car automatically. **That's the payoff of preprocessing inside the pipeline.**